# Dive into the synthesis flow

2025/09/02

Chien-Kai Ma

Quantum circuits for unambiguous quantum state discrimination as benchmarks<br>
meaningful, reproducible, and useful?

We will use three nearly-orthogonal states as example.

In [1]:
import sys
sys.path.append("../")

In [2]:
from flow.solve_mix import *
from flow.interface import *
from utils.handy_states import *
from utils.inner_product import inner_products_table

In [3]:
from flow.build_circuits import *

In [4]:
state_dict = sv_simple_2(0.2, 0.5, 0.7)
num_qubits = state_dict["num_qubits"]
num_states = state_dict["num_states"]
state_vec = state_dict["states"]
dense_mat = [DensityMatrix(_) for _ in state_vec]

In [5]:
qsd_problem = ProblemSpec(
    num_qubits=num_qubits,
    num_states=num_states,
    case_id="0902_test",
    state_type="densitymatrix",
)
n = qsd_problem.num_states
# Comment/uncomment to switch between different prior probabilities
prior_prob = np.ones(n) * (1 / n)
# prior_prob = [1/9, 3/9, 5/9]
qsd_problem.prior_prob = prior_prob

qsd_problem.set_states(
    state_type="statevector",
    states=state_vec,
    overwrite=True,
)
ideal_result = apply_Eldar(
    problem_spec=qsd_problem,
    prior_prob=prior_prob,
    cvxpy_settings={
        "solver": cp.MOSEK,
        "verbose": False,
        "eps": 1e-8,
    }
)


/home/ChienKaiMa/QSD/.venv/lib/python3.12/site-packages/mosek/__init__.py:18617: UserWarning: Argument sub in putvarboundlist: Incorrect array format causing data to be copied
  warnings.warn("Argument sub in putvarboundlist: Incorrect array format causing data to be copied");
/home/ChienKaiMa/QSD/.venv/lib/python3.12/site-packages/mosek/__init__.py:18925: UserWarning: Argument subj in putclist: Incorrect array format causing data to be copied
  warnings.warn("Argument subj in putclist: Incorrect array format causing data to be copied");
/home/ChienKaiMa/QSD/.venv/lib/python3.12/site-packages/mosek/__init__.py:18349: UserWarning: Argument sub in putconboundlist: Incorrect array format causing data to be copied
  warnings.warn("Argument sub in putconboundlist: Incorrect array format causing data to be copied");


In [6]:
qsd_problem.set_states(
    state_type="densitymatrix",
    states=dense_mat,
    overwrite=True,
)
cvxpy_med_problem = med_problem(qsd_problem, prior_prob)
eps = 1e-8
cvxpy_settings = {"solver": cp.MOSEK, "verbose": False, "eps": eps}
cvxpy_med_problem.solve(**cvxpy_settings)
vars = cvxpy_med_problem.variables()
med_povm = [var.value for var in vars]

In [7]:
def povm_to_rank1_vectors(povm, threshold=1e-4):
    """Decompose POVM elements to rank-1 vectors.
    May require using the fix function later since the output may not be POVM.
    threshold=0 means that every vector will be stored.
    threshold=1e-4 should work in most cases.
    Returns the vectors and the mapping.
    """
    povm_vectors = []
    povm_map = dict()
    idx = 0  # Index for the outcome
    for i in range(len(povm)):
        u, s, _ = np.linalg.svd(povm[i], hermitian=True)
        for j in range(len(s)):
            print(np.format_float_scientific(s[j], 4))
            if s[j] >= threshold:
                rank1_vec = u[:, j] * np.sqrt(s[j])
                povm_vectors.append(rank1_vec.conj())
                povm_map[idx] = i
                idx += 1
    return np.array(povm_vectors), povm_map

In [8]:
med_povm_vectors, _ = povm_to_rank1_vectors(med_povm, threshold=0)
med_POVM_qc = POVMCircuit(povm_vectors=med_povm_vectors)
print(len(med_povm_vectors))
print(len(med_POVM_qc.povm_vectors))
med_POVM_qc.num_qubits = num_qubits
med_POVM_qc.num_amps = 2 ** num_qubits
med_POVM_qc.case_id = f"med_nearlyOrtho_q{num_qubits}_n{num_states}"
# med_POVM_qc.fix()
med_qc = med_POVM_qc.build_circuit(scheme="csd")

1.e+00
3.3333e-01
7.9721e-09
1.9777e-10
1.e+00
3.3335e-01
5.159e-09
4.6512e-10
1.e+00
3.3333e-01
4.9649e-09
1.0013e-09
12
12


In [9]:
for item in med_povm_vectors:
    print(item)

[-0.9869-0.j  0.031 -0.j  0.0411-0.j -0.1531-0.j]
[ 0.0865-0.j  0.2164-0.j  0.3029-0.j -0.4327-0.j]
[ 9.2924e-07-0.j  6.5823e-05-0.j -5.9692e-05-0.j -8.6868e-06-0.j]
[ 8.3383e-07-0.j -7.8942e-06-0.j -7.3892e-06-0.j -8.9529e-06-0.j]
[-0.0342-0.j  0.9191-0.j -0.1075-0.j  0.3775-0.j]
[ 0.0866-0.j  0.2164-0.j  0.3029-0.j -0.4327-0.j]
[ 6.2027e-05-0.j  2.5769e-06-0.j -3.4565e-05-0.j -1.0501e-05-0.j]
[ 1.0357e-05-0.j -2.5030e-06-0.j  1.4966e-05-0.j  1.1296e-05-0.j]
[-0.0495-0.j -0.1175-0.j  0.8435-0.j  0.5218-0.j]
[-0.0865-0.j -0.2164-0.j -0.3029-0.j  0.4327-0.j]
[-6.1075e-05-0.j  3.4854e-05-0.j -1.3661e-06-0.j  4.2554e-06-0.j]
[-1.4969e-05-0.j -2.4533e-05-0.j  3.5902e-06-0.j -1.2747e-05-0.j]


In [10]:
print("MED POVM QC")
print("num_qubits", med_qc.decompose(reps=3).num_qubits)
print("depth", med_qc.decompose(reps=3).depth())
print("gates", med_qc.decompose(reps=3).count_ops())

MED POVM QC
num_qubits 4
depth 158
gates OrderedDict({'u': 148, 'cx': 68})


In [37]:
reduced_med_povm_vectors, _ = povm_to_rank1_vectors(med_povm, threshold=1e-4)
# reduced_med_povm_vectors, _ = povm_to_rank1_vectors(med_povm, threshold=5e-1)

reduced_med_POVM_qc = POVMCircuit(povm_vectors=reduced_med_povm_vectors)

print(verify_povm(reduced_med_povm_vectors))
print(len(reduced_med_povm_vectors))
print(len(reduced_med_POVM_qc.povm_vectors))
reduced_med_POVM_qc.num_qubits = num_qubits
reduced_med_POVM_qc.num_amps = 2 ** num_qubits
reduced_med_POVM_qc.case_id = f"noisy_med_nearlyOrtho_q{num_qubits}_n{num_states}"
reduced_med_POVM_qc.fix()
print(len(reduced_med_POVM_qc.povm_vectors))
reduced_med_qc = reduced_med_POVM_qc.build_circuit(scheme="csd")

1.e+00
3.3333e-01
1.7230e-08
1.775e-09
1.e+00
3.3334e-01
1.0339e-08
2.7661e-09
1.e+00
3.3333e-01
9.8353e-09
3.6355e-09
<class 'numpy.ndarray'>
False
6
6
6


In [36]:
print("Reduced MED POVM QC")
print("num_qubits", reduced_med_qc.decompose(reps=3).num_qubits)
print("depth", reduced_med_qc.decompose(reps=3).depth())
print("gates", reduced_med_qc.decompose(reps=3).count_ops())

Reduced MED POVM QC
num_qubits 3
depth 156
gates OrderedDict({'u': 121, 'cx': 68, 'u3': 20, 'ry': 10, 'u1': 10})


In [13]:
obj = POVMCircuit(povm_vectors=ideal_result["povm_vectors"])
obj.num_qubits = num_qubits
obj.num_amps = 2 ** num_qubits
obj.case_id = f"nearlyOrtho_q{num_qubits}_n{num_states}"
obj.fix()
optuqsd_qc = obj.build_circuit()

In [14]:
for item in ideal_result["povm_vectors"]:
    print(item)
optuqsd_povm = vectors_to_povm(ideal_result["povm_vectors"])
for item in optuqsd_povm:
    print(item)

[ 0.9775+0.j -0.0562-0.j -0.0787-0.j  0.1124+0.j]
[-0.0562-0.j  0.8596+0.j -0.1966-0.j  0.2809+0.j]
[-0.0787-0.j -0.1966-0.j  0.7247+0.j  0.3933+0.j]
[[ 0.9556+0.j -0.0549+0.j -0.0769+0.j  0.1098+0.j]
 [-0.0549+0.j  0.0032+0.j  0.0044+0.j -0.0063+0.j]
 [-0.0769+0.j  0.0044+0.j  0.0062+0.j -0.0088+0.j]
 [ 0.1098+0.j -0.0063+0.j -0.0088+0.j  0.0126+0.j]]
[[ 0.0032+0.j -0.0483+0.j  0.011 +0.j -0.0158+0.j]
 [-0.0483+0.j  0.7388+0.j -0.169 +0.j  0.2414+0.j]
 [ 0.011 +0.j -0.169 +0.j  0.0387+0.j -0.0552+0.j]
 [-0.0158+0.j  0.2414+0.j -0.0552+0.j  0.0789+0.j]]
[[ 0.0062+0.j  0.0155+0.j -0.057 +0.j -0.0309+0.j]
 [ 0.0155+0.j  0.0387+0.j -0.1425+0.j -0.0773+0.j]
 [-0.057 +0.j -0.1425+0.j  0.5252+0.j  0.285 +0.j]
 [-0.0309+0.j -0.0773+0.j  0.285 +0.j  0.1547+0.j]]


In [15]:
print("OptUQSD POVM QC without noise")
print("num_qubits", optuqsd_qc.decompose(reps=3).num_qubits)
print("depth", optuqsd_qc.decompose(reps=3).depth())
print("gates", optuqsd_qc.decompose(reps=3).count_ops())

OptUQSD POVM QC without noise
num_qubits 3
depth 83
gates OrderedDict({'u': 64, 'cx': 41, 'rz': 6})


In [16]:
# Evaluate with direct measurement
sum = 0
for i in range(len(state_vec)):
    sum += prior_prob[i] * np.abs(state_vec[i].data[i] ** 2)
print(sum)

0.8108931337119256


In [17]:
ideal_result["p_succ"]

np.float64(0.8108931335061406)

In fact, measurement in the basis states is actually UQSD, and it yields the optimal $P_\text{succ}$!

In [18]:
# Show index mapping

In [19]:
# med_qc.draw(style="mpl")

In [20]:
noisy_qsd_problem = ProblemSpec(
    num_qubits=num_qubits,
    num_states=num_states,
    case_id="0902_test",
    state_type="densitymatrix",
)

noisy_qsd_problem.prior_prob = prior_prob

disturbance_states = [
    DensityMatrix(
        ProblemSpec.depolarizing_noise_channel(num_qubits=num_qubits)
    )
    for _ in range(num_states)
]

noise_level = 0.1
combined_states = [
    (1 - noise_level) * dense_mat[_] + noise_level * disturbance_states[_].data
    for _ in range(num_states)
]

noisy_qsd_problem.set_states(
    state_type="densitymatrix",
    states=combined_states,
    overwrite=True,
)

cvxpy_noisy_med_problem = med_problem(noisy_qsd_problem, prior_prob)

In [21]:
def run_qsd(
    qsd_problem: ProblemSpec,
    cvxpy_problem,
    jsd_result,
    sqrtd_result,
    psucc_result,
    eps=1e-8,
):
    cvxpy_settings = {"solver": cp.MOSEK, "verbose": False, "eps": eps}
    cvxpy_problem.solve(**cvxpy_settings)

    # print(cvxpy_problem.status)
    # print(cvxpy_problem.solution.opt_val)
    vars = cvxpy_problem.variables()

    fitqd_povm = [var.value for var in vars]
    # print(fitqd_povm)

    prob_mat = calculate_prob_matrix_simple(
        prior_probs=prior_prob,
        povm=fitqd_povm,
        states=qsd_problem.states,
    )
    # print(prob_mat)

    psucc = 0
    for i in range(qsd_problem.num_states):
        psucc += prob_mat[i][i]
    psucc_result.append(psucc)

    js_dist = jensenshannon(
        np.array(ideal_distrib).flatten(),
        np.array(prob_mat).flatten(),
    )
    jsd_result.append(js_dist)

    sqrt_dist = get_sqrt_dist(
        np.array(ideal_distrib).flatten(),
        np.array(prob_mat).flatten(),
    )
    sqrtd_result.append(sqrt_dist)
    #
    # print("alpha", np.array(calculate_errors(prob_mat)[0]))
    # print("beta ", max(calculate_errors(prob_mat)[1]))
    #
    # print()

In [22]:
eps = 1e-8
cvxpy_settings = {"solver": cp.MOSEK, "verbose": False, "eps": eps}
cvxpy_noisy_med_problem.solve(**cvxpy_settings)

np.float64(0.9207136780240412)

In [23]:
vars = cvxpy_noisy_med_problem.variables()
med_povm = [var.value for var in vars]

In [24]:
for item in med_povm:
    print(item)

[[ 0.9814+0.j -0.0118+0.j -0.0143+0.j  0.1137+0.j]
 [-0.0118+0.j  0.0478+0.j  0.0668+0.j -0.0984+0.j]
 [-0.0143+0.j  0.0668+0.j  0.0934+0.j -0.1374+0.j]
 [ 0.1137+0.j -0.0984+0.j -0.1374+0.j  0.2107+0.j]]
[[ 0.0087+0.j -0.0127+0.j  0.0299+0.j -0.0504+0.j]
 [-0.0127+0.j  0.8916+0.j -0.0333+0.j  0.2533+0.j]
 [ 0.0299+0.j -0.0333+0.j  0.1033+0.j -0.1717+0.j]
 [-0.0504+0.j  0.2533+0.j -0.1717+0.j  0.3297+0.j]]
[[ 0.0099+0.j  0.0245+0.j -0.0156+0.j -0.0633+0.j]
 [ 0.0245+0.j  0.0606+0.j -0.0335+0.j -0.1549+0.j]
 [-0.0156+0.j -0.0335+0.j  0.8032+0.j  0.3091+0.j]
 [-0.0633+0.j -0.1549+0.j  0.3091+0.j  0.4595+0.j]]


In [25]:
verify_povm_matrix(med_povm)

True

In [26]:
# TODO Check old code
"""
# Check the rank of the Hermitian operators
# Log the rank of the Hermitian operators
# Dictionary of measured bitstrings to the target states
"""

'\n# Check the rank of the Hermitian operators\n# Log the rank of the Hermitian operators\n# Dictionary of measured bitstrings to the target states\n'

In [27]:
reduced_noisy_med_povm_vectors, _ = povm_to_rank1_vectors(med_povm, threshold=1e-4)

reduced_noisy_med_POVM_qc = POVMCircuit(povm_vectors=reduced_noisy_med_povm_vectors)
print()
print(verify_povm(reduced_noisy_med_povm_vectors))
print(len(reduced_noisy_med_povm_vectors))
print(len(reduced_noisy_med_POVM_qc.povm_vectors))
reduced_noisy_med_POVM_qc.num_qubits = num_qubits
reduced_noisy_med_POVM_qc.num_amps = 2 ** num_qubits
reduced_noisy_med_POVM_qc.case_id = f"noisy_med_nearlyOrtho_q{num_qubits}_n{num_states}"
reduced_noisy_med_POVM_qc.fix()
print(len(reduced_noisy_med_POVM_qc.povm_vectors))
reduced_noisy_med_qc = reduced_noisy_med_POVM_qc.build_circuit()

1.e+00
3.3333e-01
1.7230e-08
1.775e-09
1.e+00
3.3334e-01
1.0339e-08
2.7661e-09
1.e+00
3.3333e-01
9.8353e-09
3.6355e-09

<class 'numpy.ndarray'>
False
6
6
6


In [28]:
print("Reduced Noisy MED POVM QC")
print("num_qubits", reduced_noisy_med_qc.decompose(reps=3).num_qubits)
print("depth", reduced_noisy_med_qc.decompose(reps=3).depth())
print("gates", reduced_noisy_med_qc.decompose(reps=3).count_ops())

Reduced Noisy MED POVM QC
num_qubits 3
depth 84
gates OrderedDict({'u': 64, 'cx': 41, 'rz': 7})


In [29]:
noisy_med_povm_vectors, _ = povm_to_rank1_vectors(med_povm, threshold=0)
ops = []
for m in noisy_med_povm_vectors:
    print(m)
    # Add "None" to transpose
    # https://stackoverflow.com/a/11885718/13518808
    op = np.multiply(m[None].T.conj(), m)
    ops.append(op)
# print(ops)
verify_povm(noisy_med_povm_vectors, rtol=0.00001)

1.e+00
3.3333e-01
1.7230e-08
1.775e-09
1.e+00
3.3334e-01
1.0339e-08
2.7661e-09
1.e+00
3.3333e-01
9.8353e-09
3.6355e-09
[-0.9869-0.j  0.031 -0.j  0.0411-0.j -0.1531-0.j]
[ 0.0866-0.j  0.2164-0.j  0.3029-0.j -0.4327-0.j]
[ 1.2891e-06-0.j  9.7492e-05-0.j -8.7070e-05-0.j -1.1945e-05-0.j]
[ 2.5021e-06-0.j -2.3342e-05-0.j -2.2414e-05-0.j -2.6861e-05-0.j]
[-0.0342-0.j  0.9191-0.j -0.1075-0.j  0.3775-0.j]
[ 0.0866-0.j  0.2164-0.j  0.3029-0.j -0.4327-0.j]
[-8.9731e-05-0.j -3.1627e-06-0.j  4.6008e-05-0.j  1.2678e-05-0.j]
[-2.3379e-05-0.j  6.1761e-06-0.j -3.7501e-05-0.j -2.7839e-05-0.j]
[-0.0495-0.j -0.1175-0.j  0.8435-0.j  0.5218-0.j]
[-0.0866-0.j -0.2164-0.j -0.3029-0.j  0.4327-0.j]
[-8.6490e-05-0.j  4.8177e-05-0.j -1.7944e-06-0.j  5.5342e-06-0.j]
[-2.7925e-05-0.j -4.7083e-05-0.j  6.8541e-06-0.j -2.4330e-05-0.j]
<class 'numpy.ndarray'>


True

In [30]:
print(len(ideal_result["povm"]))
print(len(obj.povm_vectors))
for item in obj.povm_vectors:
    print(item)


4
5
[ 0.9775+0.j -0.0562-0.j -0.0787-0.j  0.1124+0.j]
[-0.0562-0.j  0.8596+0.j -0.1966-0.j  0.2809+0.j]
[-0.0787-0.j -0.1966-0.j  0.7247+0.j  0.3933+0.j]
[-0.1499-0.j -0.3748-0.j -0.5247-0.j  0.7495-0.j]
[-0.1124-0.j -0.2809-0.j -0.3933-0.j -0.4382-0.j]


In [31]:
noisy_med_POVM_qc = POVMCircuit(povm_vectors=noisy_med_povm_vectors)
print(len(noisy_med_povm_vectors))
print(len(noisy_med_POVM_qc.povm_vectors))
noisy_med_POVM_qc.num_qubits = num_qubits
noisy_med_POVM_qc.num_amps = 2 ** num_qubits
noisy_med_POVM_qc.case_id = f"noisy_med_nearlyOrtho_q{num_qubits}_n{num_states}"
# noisy_med_POVM_qc.fix()
noisy_med_qc = noisy_med_POVM_qc.build_circuit()

12
12


In [32]:
print("Noisy MED POVM QC")
print("num_qubits", noisy_med_qc.decompose(reps=3).num_qubits)
print("depth", noisy_med_qc.decompose(reps=3).depth())
print("gates", noisy_med_qc.decompose(reps=3).count_ops())

Noisy MED POVM QC
num_qubits 4
depth 430
gates OrderedDict({'u': 285, 'cx': 218, 'rz': 15})
